# Regression — Appliances Energy Prediction
Este notebook resolve a tarefa de regressão indicada no enunciado: prever o consumo de energia (Wh) dos eletrodomésticos a partir de variáveis ambientais.
**Objetivos:** EDA, pré-processamento, treinamento de modelos (Regressão Linear, Decision Tree Regressor, Random Forest Regressor), avaliação (R², RMSE, MAE) e comparação de resultados.


## 1. Preparação (instalação e imports)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

DATA_PATH = Path('../data')
appliances_csv_candidates = list(DATA_PATH.rglob('*.csv'))
print('CSV candidates inside ../data:', appliances_csv_candidates)


## 2. Carregar dataset

In [ ]:
assert appliances_csv_candidates, 'Coloque o CSV do dataset em `deliverable_repo/data/` e execute novamente.'
df = pd.read_csv(appliances_csv_candidates[0])
print('Dataset carregado:', appliances_csv_candidates[0])
display(df.head())
print('\nDimensões do dataset:', df.shape)


## 3. Exploração rápida (EDA)

In [ ]:
print(df.info())
print('\nDescrição estatística:')
display(df.describe().T)
print('\nValores ausentes por coluna:')
display(df.isna().sum())


## 4. Pré-processamento (exemplo)
Exemplo de passos: converter colunas de data/hora, extrair features temporais, normalizar atributos contínuos, selecionar features relevantes, lidar com outliers.


In [ ]:
# Exemplo genérico — adapte conforme as colunas reais do seu CSV
df2 = df.copy()
for col in df2.columns:
    if 'date' in col.lower() or 'time' in col.lower():
        try:
            df2[col] = pd.to_datetime(df2[col])
        except Exception:
            pass

# Exemplo: criar features temporais se houver datetime
datetime_cols = [c for c in df2.columns if pd.api.types.is_datetime64_any_dtype(df2[c])]
if datetime_cols:
    dtc = datetime_cols[0]
    df2['hour'] = df2[dtc].dt.hour
    df2['dayofweek'] = df2[dtc].dt.dayofweek

# Selecionar target e features (ajuste conforme seu dataset real)
possible_targets = [c for c in df2.columns if 'wh' in c.lower() or 'appliances' in c.lower() or 'energy' in c.lower()]
print('Possible targets found:', possible_targets)
if possible_targets:
    target = possible_targets[0]
else:
    # fallback: last numeric column
    numeric_cols = df2.select_dtypes(include=[np.number]).columns.tolist()
    target = numeric_cols[-1]

features = df2.select_dtypes(include=[np.number]).columns.drop(target).tolist()
print('\nTarget:', target)
print('Num features:', len(features))


## 5. Treinamento e avaliação de modelos

In [ ]:
# Preparar X e y
X = df2[features].fillna(method='ffill').fillna(0)
y = df2[target].fillna(method='ffill').fillna(0)

# Separar treino/teste (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Padronizar (opcional)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Modelos a treinar
models = {
    'LinearRegression': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []
for name, m in models.items():
    print('\nTreinando', name)
    # para LinearRegression usar dados padronizados; para árvores não é necessário
    if name == 'LinearRegression':
        m.fit(X_train_s, y_train)
        preds = m.predict(X_test_s)
    else:
        m.fit(X_train, y_train)
        preds = m.predict(X_test)
    r2 = r2_score(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)
    mae = mean_absolute_error(y_test, preds)
    print(f'{name} — R2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}')
    results.append({'model': name, 'r2': r2, 'rmse': rmse, 'mae': mae})

res_df = pd.DataFrame(results).sort_values('r2', ascending=False)
display(res_df)


## 6. Discussão e próximos passos
- Compare modelos usando R², RMSE e MAE.
- Explorar tuning de hiperparâmetros (GridSearchCV/RandomizedSearchCV).
- Tentar features adicionais (lags, rolling statistics) e engenharia temporal.
- Se a série for temporal dependente, considerar modelos de séries temporais / modelos com janela (LSTM, XGBoost com features lagged, etc.).
